# v19 최종 선택 모델 — 전체 train 재학습 및 제출 후보 생성

개발 검증으로 고정한 설정입니다. test는 transform/predict에만 사용합니다. 모델 또는 앙상블 선택에 test/holdout 정답을 사용하지 않습니다. 학습 코드와 원본 복사 노트북을 함께 보존하세요.

```json
{
  "best": {
    "version": "v17_without_combos",
    "alpha": 1.0,
    "C": 0.1,
    "mode": "base",
    "ablate": "USE_COMBOS",
    "xgb_weight": 0.4,
    "cv_macro_f1": 0.4694511564910411,
    "seconds": 29.9
  },
  "seed_ensemble": true,
  "seed_ensemble_cv": 0.46959099957995554,
  "selection_uses": "development CV only; holdout not evaluated yet"
}
```

Public 0.47959 초과 여부는 아직 알 수 없습니다. 확인용 점수는 RESULTS.md에 기록합니다.


In [ ]:
from pathlib import Path
import json
HERE=Path.cwd()
if not (HERE/'run_experiments.py').exists():
    HERE=HERE/'models'/'혜진'/'codex'/'minjung_experiments'
script=HERE/'run_experiments.py'
selection=json.loads((HERE/'frozen_selection.json').read_text())
# Freeze model settings before reading test.
cfg=selection['best']
ns={'__file__':str(script.resolve())}
exec(compile(script.read_text().split('# Query transforms may see')[0],str(script),'exec'),ns)
np=ns['np']; pd=ns['pd']; DATA=ns['DATA']
test=pd.read_csv(DATA/'test.csv').fillna('WT')
sub=pd.read_csv(DATA/'sample_submission.csv')
assert list(test.columns)==['ID']+ns['gene_cols'] and np.array_equal(test.ID,sub.ID)
query=ns['extract'](test); query_docs=ns['token_docs'](test)
rows=np.arange(len(ns['y'])); w=cfg['xgb_weight']
px,pl=ns['fit_predict'](rows,query,query_docs,cfg,seed=42)
P=w*px+(1-w)*pl
if selection['seed_ensemble']:
    px,pl=ns['fit_predict'](rows,query,query_docs,cfg,seed=137)
    P=.5*P+.5*(w*px+(1-w)*pl)
sub['SUBCLASS']=ns['le'].inverse_transform(P.argmax(1))
sub.to_csv(HERE/'submission_selected_core_reproduced.csv',index=False)
print('Saved',len(sub),'predictions; no competition submission performed.')
